It is highly recommended to use a powerful **GPU**, you can use it for free uploading this notebook to [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb).
<table align="center">
 <td align="center"><a target="_blank" href="https://colab.research.google.com/github/ezponda/intro_deep_learning/blob/main/class/CNN/Open_Vocabulary_Detection.ipynb">
        <img src="https://colab.research.google.com/img/colab_favicon_256px.png"  width="50" height="50" style="padding-bottom:5px;" />Run in Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/ezponda/intro_deep_learning/blob/main/class/CNN/Open_Vocabulary_Detection.ipynb">
        <img src="https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png"  width="50" height="50" style="padding-bottom:5px;" />View Source on GitHub</a></td>
</table>

**Table of Contents**

1. [The Vocabulary Problem](#vocabulary-problem)
2. [Introduction to Grounding DINO](#grounding-dino)
3. [Basic Inference](#basic-inference)
4. [YOLO vs Grounding DINO](#yolo-vs-gdino)
5. [Beyond Fixed Categories](#beyond-fixed)
6. [Grounded-SAM: Text-Prompted Segmentation](#grounded-sam)

<a id='vocabulary-problem'></a>
# The Vocabulary Problem

Traditional object detectors like YOLO are trained on a fixed set of categories. YOLO models trained on the [COCO dataset](https://cocodataset.org/) can detect exactly 80 classes: person, car, dog, cat, bicycle, etc.

But what if you need to detect something more specific?

- "red car" vs "blue car" -- YOLO only knows "car"
- "person wearing a helmet" -- YOLO only knows "person"
- "golden retriever" vs "bulldog" -- YOLO only knows "dog"
- "fire hydrant that is damaged" -- YOLO cannot understand attributes
- "tuk-tuk", "segway", "electric scooter" -- not in COCO's 80 classes

Adding new categories to YOLO requires collecting labeled data and retraining. **Open-vocabulary detection** solves this by using natural language: you describe what you want to find, and the model detects it.

<a id='grounding-dino'></a>
# Introduction to Grounding DINO

[Grounding DINO](https://arxiv.org/abs/2303.05499) (ECCV 2024) is an open-vocabulary object detector that combines a visual encoder with a text encoder. Instead of a fixed classification head with N output neurons, it computes similarity between visual features and text tokens.

### Architecture Overview

The model has three main components:

1. **Image encoder** (Swin Transformer): Extracts multi-scale visual features from the image.
2. **Text encoder** (BERT): Encodes the text prompt into token embeddings. Categories are separated by periods (e.g., `"red car. person with hat. dog."`).
3. **Cross-modal fusion**: Multiple layers of bidirectional attention between image and text features. The image features learn *where* the text concepts appear, and the text features learn *what* visual patterns correspond to each word.

The detection head computes a similarity score between each candidate region and each text token. If the similarity exceeds a threshold, the region is detected as that category.

### Key Difference from YOLO

| | YOLO | Grounding DINO |
|---|---|---|
| **Vocabulary** | Fixed (80 COCO classes) | Any text prompt |
| **Adding new classes** | Requires retraining | Just change the prompt |
| **Speed** | Very fast (~5ms) | Slower (~200ms) |
| **Best for** | Known categories, real-time | Novel categories, flexible queries |

You can install the required packages with:

```python
!pip install transformers accelerate
!pip install ultralytics
```

In [ ]:
#!pip install transformers accelerate
#!pip install ultralytics

<a id='basic-inference'></a>
# Basic Inference

Let's load Grounding DINO and detect objects using text prompts. Categories in the prompt are separated by periods.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import urllib

# Load Grounding DINO (tiny variant, ~3-4 GB VRAM)
model_id = "IDEA-Research/grounding-dino-tiny"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)
print(f"Model loaded on {device}")

In [ ]:
def detect(image_pil, text_prompt, threshold=0.3):
    """Run Grounding DINO on an image with a text prompt."""
    inputs = processor(images=image_pil, text=text_prompt,
                       return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = gd_model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs, threshold=threshold,
        target_sizes=[(image_pil.height, image_pil.width)]
    )[0]
    return results


def show_detections(image_pil, results, title="Detections"):
    """Visualize detection results on an image."""
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(image_pil)

    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    labels = results["text_labels"]

    colors = plt.cm.Set2(np.linspace(0, 1, max(len(boxes), 1)))
    for box, score, label, color in zip(boxes, scores, labels, colors):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-8, f'{label} ({score:.2f})',
                color='white', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85))

    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    return len(boxes)

In [ ]:
# Download sample image
url = 'https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/traffic.jpg'
urllib.request.urlretrieve(url, 'traffic.jpg')
image = Image.open('traffic.jpg').convert("RGB")

# Detect with a text prompt (categories separated by periods)
text_prompt = "car. person. bus. truck."
results = detect(image, text_prompt)

print(f"Detected {len(results['boxes'])} objects")
show_detections(image, results, title=f'Prompt: "{text_prompt}"')

Now try something YOLO cannot do -- detect with **attributes**:

In [ ]:
# Detect specific attributes
text_prompt = "red car. white truck. traffic light."
results = detect(image, text_prompt, threshold=0.25)

show_detections(image, results, title=f'Attribute-Based: "{text_prompt}"')

<a id='yolo-vs-gdino'></a>
# YOLO vs Grounding DINO

Let's compare both models side-by-side on the same images. YOLO is fast but limited to 80 classes. Grounding DINO is slower but can detect anything you describe.

In [ ]:
from ultralytics import YOLO

yolo = YOLO('yolo11n.pt')

In [ ]:
def compare_yolo_gdino(image_path, gdino_prompt, threshold=0.3):
    """Compare YOLO and Grounding DINO on the same image."""
    image_pil = Image.open(image_path).convert("RGB")
    image_np = np.array(image_pil)
    
    # YOLO detection
    yolo_results = yolo(image_np, verbose=False)[0]
    yolo_annotated = yolo_results.plot()[..., ::-1]  # BGR to RGB
    
    # Grounding DINO detection
    gdino_results = detect(image_pil, gdino_prompt, threshold=threshold)
    
    # Plot side by side
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    axes[0].imshow(yolo_annotated)
    axes[0].set_title(f'YOLO (80 fixed classes)\n{len(yolo_results.boxes)} detections')
    axes[0].axis('off')
    
    axes[1].imshow(image_pil)
    boxes = gdino_results["boxes"].cpu().numpy()
    scores = gdino_results["scores"].cpu().numpy()
    labels = gdino_results["text_labels"]
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(boxes), 1)))
    for box, score, label, color in zip(boxes, scores, labels, colors):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        axes[1].add_patch(rect)
        axes[1].text(x1, y1-8, f'{label} ({score:.2f})',
                     color='white', fontsize=9, fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85))
    axes[1].set_title(f'Grounding DINO: "{gdino_prompt}"\n{len(boxes)} detections')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Standard categories -- both models work well
compare_yolo_gdino('traffic.jpg', "car. person. bus. truck.", threshold=0.3)

In [ ]:
# Attribute-rich prompt -- only Grounding DINO can handle this
compare_yolo_gdino('traffic.jpg', "red car. white truck. pedestrian crossing the road.", threshold=0.25)

In [ ]:
# Download images used in the YOLO notebook
image_urls = [
    ('https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/beach.jpg', 'beach.jpg'),
    ('https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images/dog_cat.jpg', 'dog_cat.jpg'),
]
for url, path in image_urls:
    urllib.request.urlretrieve(url, path)

compare_yolo_gdino('beach.jpg', "person. surfboard. umbrella.", threshold=0.25)

In [ ]:
compare_yolo_gdino('dog_cat.jpg', "dog. cat.", threshold=0.2)

<a id='beyond-fixed'></a>
# Beyond Fixed Categories

Grounding DINO's power comes from understanding natural language descriptions. Let's explore what kind of prompts work:

In [ ]:
image = Image.open('traffic.jpg').convert("RGB")

prompts = [
    "vehicle",                      # generic
    "car. truck. bus.",             # specific vehicle types
    "wheel",                        # parts of objects
    "road. sidewalk. building.",    # scene elements
    "traffic sign. traffic light.", # infrastructure
]

fig, axes = plt.subplots(len(prompts), 1, figsize=(14, 6 * len(prompts)))

for ax, prompt in zip(axes, prompts):
    results = detect(image, prompt, threshold=0.25)
    
    ax.imshow(image)
    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    labels = results["text_labels"]
    colors_list = plt.cm.Set2(np.linspace(0, 1, max(len(boxes), 1)))
    
    for box, score, label, color in zip(boxes, scores, labels, colors_list):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, f'{label} ({score:.2f})',
                color='white', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))
    ax.set_title(f'Prompt: "{prompt}" -- {len(boxes)} detections', fontsize=13)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Question 1: Prompt Engineering

Choose one of the images (traffic, beach, or dog_cat) and write prompts to detect:
1. A very generic concept (e.g., "object", "animal")
2. Specific subcategories (e.g., "golden retriever", "tabby cat")
3. Attributes (e.g., "large dog", "person wearing red")
4. Scene elements (e.g., "shadow", "reflection", "sky")

Compare the results. Which prompts work well? Which ones fail?

In [ ]:
image = Image.open(...).convert("RGB")

# Try your prompts here
prompt_1 = ...  # generic
prompt_2 = ...  # specific subcategories
prompt_3 = ...  # attributes
prompt_4 = ...  # scene elements

for prompt in [prompt_1, prompt_2, prompt_3, prompt_4]:
    results = detect(image, prompt, threshold=0.25)
    n = show_detections(image, results, title=f'Prompt: "{prompt}"')
    print(f'  -> {n} detections\n')

## Question 2: Object Counting with Text Prompts

Use Grounding DINO to count specific objects in images. Try to count:
1. Total number of people on the beach image
2. Number of cars vs trucks on the traffic image
3. Try counting a specific subset: "person in the water" vs "person on the sand"

Compare your text-based counts with YOLO's detection counts.

In [ ]:
image = Image.open('beach.jpg').convert("RGB")

# Count people
results_people = detect(image, ..., threshold=...)
print(f"People detected: {len(results_people['boxes'])}")

# Count a specific subset
results_subset = detect(image, ..., threshold=...)
print(f"Subset detected: {len(results_subset['boxes'])}")

# Compare with YOLO
yolo_results = yolo(np.array(image), verbose=False)[0]
yolo_people = sum(1 for cls in yolo_results.boxes.cls if int(cls) == 0)
print(f"YOLO people count: {yolo_people}")

<a id='grounded-sam'></a>
# Grounded-SAM: Text-Prompted Segmentation

We can combine Grounding DINO with Meta's [Segment Anything Model (SAM)](https://segment-anything.com/) to go from text prompts to pixel-perfect segmentation masks:

1. **Grounding DINO** produces bounding boxes from text
2. **SAM** produces precise segmentation masks from bounding boxes

This pipeline is known as **Grounded-SAM** and enables text-to-segmentation without any training.

In [ ]:
from transformers import SamModel, SamProcessor

sam_model_id = "facebook/sam-vit-base"
sam_model = SamModel.from_pretrained(sam_model_id).to(device)
sam_processor = SamProcessor.from_pretrained(sam_model_id)
print("SAM loaded")

In [ ]:
def grounded_sam(image_pil, text_prompt, threshold=0.3):
    """Text prompt -> bounding boxes (Grounding DINO) -> masks (SAM)."""
    # Step 1: Get boxes from Grounding DINO
    gd_results = detect(image_pil, text_prompt, threshold)
    boxes = gd_results["boxes"].cpu().tolist()
    labels = gd_results["text_labels"]
    
    if len(boxes) == 0:
        print("No detections found.")
        return None, gd_results, labels
    
    # Step 2: Feed boxes to SAM as prompts
    inputs = sam_processor(image_pil, input_boxes=[boxes],
                           return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = sam_model(**inputs)
    
    masks = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )[0]  # (N, 3, H, W) -- take best mask per object (index 0)
    
    return masks, gd_results, labels

In [ ]:
image = Image.open('traffic.jpg').convert("RGB")
text_prompt = "car. truck."

masks, gd_results, labels = grounded_sam(image, text_prompt, threshold=0.3)

# Visualize: overlay colored masks on the image
img_array = np.array(image).copy()
colors_mask = plt.cm.Set2(np.linspace(0, 1, len(masks)))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(image)
axes[0].set_title('Original')
axes[0].axis('off')

overlay = img_array.astype(np.float32)
for i, (mask, label) in enumerate(zip(masks, labels)):
    # Take the best mask (index 0) and squeeze
    binary_mask = mask[0].numpy().squeeze() > 0.5
    color = np.array(colors_mask[i][:3]) * 255
    overlay[binary_mask] = overlay[binary_mask] * 0.5 + color * 0.5

axes[1].imshow(overlay.astype(np.uint8))
axes[1].set_title(f'Grounded-SAM: "{text_prompt}"')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Question 3: Text-Prompted Segmentation

Use Grounded-SAM to segment specific objects:
1. Segment all people on the beach image
2. Segment only the sky or the sea
3. Try segmenting something very specific: "surfboard", "umbrella"

Visualize the masks overlaid on the original image.

In [ ]:
image = Image.open('beach.jpg').convert("RGB")

# Segment specific objects
text_prompt = ...
masks, gd_results, labels = grounded_sam(image, text_prompt, threshold=...)

# Visualize
img_array = np.array(image).copy().astype(np.float32)
colors_mask = plt.cm.Set2(np.linspace(0, 1, max(len(masks), 1)))

for i, (mask, label) in enumerate(zip(masks, labels)):
    binary_mask = mask[0].numpy().squeeze() > 0.5
    color = np.array(colors_mask[i][:3]) * 255
    img_array[binary_mask] = img_array[binary_mask] * 0.5 + color * 0.5

plt.figure(figsize=(12, 8))
plt.imshow(img_array.astype(np.uint8))
plt.title(f'Grounded-SAM: "{text_prompt}"')
plt.axis('off')
plt.show()

## Question 4: Background Removal with Text

Use Grounded-SAM to isolate a specific object from the background:
1. Detect and segment the object with a text prompt
2. Set all background pixels (outside the mask) to white
3. Save the result as a new image

This is the same technique used by tools like remove.bg, but driven by a text prompt.

In [ ]:
image = Image.open(...).convert("RGB")

# Step 1: Segment the target object
text_prompt = ...
masks, gd_results, labels = grounded_sam(image, text_prompt, threshold=...)

# Step 2: Create combined mask for all detections
img_array = np.array(image).copy()
combined_mask = np.zeros(img_array.shape[:2], dtype=bool)
for mask in masks:
    binary_mask = mask[0].numpy().squeeze() > 0.5
    combined_mask = combined_mask | binary_mask

# Step 3: Set background to white
result = img_array.copy()
result[~combined_mask] = ...  # Set to white (255, 255, 255)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(image)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(result)
axes[1].set_title(f'Isolated: "{text_prompt}"')
axes[1].axis('off')
plt.tight_layout()
plt.show()